# Step 23.4 — Generation module

**命令一：**

~~~bash
git pull
~~~

**命令二：**

~~~bash
cp -n notebooks/105_step23_4_generation_module.ipynb notebooks/work/105_step23_4_generation_module_$(date +%Y%m%d_%H%M%S)_work.ipynb
~~~

只在独立 work 副本运行/修改；本轮只静态准备，未执行任何 Notebook、MLX、训练、生成或 benchmark。Apple Silicon / Python 3.12 / MLX v0.32.2。


## 1. 动机（Motivation）

Step 21.1/21.5 的生成循环仍留在 Notebook 中。本节把 greedy 与按温度的 categorical sampling 变成可导入的两个函数，保持已验收模型无 cache 的行为；新的职责是封装输入检查、窗口切片、显式随机 key 与推理模式恢复，不是训练或发明新解码架构。

完整运行实现放在 **src/minigpt/generate.py**，以下 Notebook 只教学本小节新增接口，不复制旧类的长代码。


## 2. Shape / Input–Output Contract

模型读取 context (1,min(L+r,W))，返回 raw logits (1,T,V)。取 logits[:, -1, :] -> last (1,V)，greedy argmax 或 sample 选择 token (1,)，token[:,None] -> (1,1)，沿 axis=1 拼到完整 history (1,L+r+1)。返回的 greedy history (1,L+n)；sampling 额外返回 next_key (2,) uint32（固定 MLX key 约定）。W<=model.max_seq_len；n=0 无 forward。


## 3. 手算小例子（Paper example, not runtime output）

prompt IDs=[0,1]、block_size=3，追加 3 个 token。三轮 forward 的 context 长度为 2、3、3；最终 history 长 5。不预言具体 token，因为真实模型和参数未知。对于已知 logits [0, ln2]，greedy 选 ID 1，按温度 1 抽样则理论概率为 [1/3,2/3]；实际 draw 不填写。


In [ ]:
# Future independent work copy only. No generation here.
# 从 root/notebooks/notebooks/work 向上定位源包。
from pathlib import Path
import sys

repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file()
     and (p / "src" / "minigpt" / "__init__.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("MiniGPT repository not found above kernel working directory")
src_dir = str(repo_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
from minigpt.generate import generate_greedy, generate_sample
# ids = generate_greedy(model, [0,1], 3, block_size=3)
# sampled_ids, next_key = generate_sample(
#     model, [0,1], 3, block_size=3, temperature=1.0, key=key)
# model and key must be supplied by the future user; nothing runs now.


## 4. 限制、预期验证与静态结论

Crop 保留返回历史但每轮重新计算整个窗口，**没有 KV cache**。旧模型 learned position 在新窗口从 0 编号，不能无损复用旧 K/V。该模块目前只处理单条 prompt、固定 n 个 token；没有 EOS、batch、top-k 或 top-p；也未实现 GPU fused sampling。Sampling 每轮 split 显式 key，不使用隐式全局 key；温度必须为正有限。

未来可验证的前提是合适的真实模型、正确的数组和显式调用；静态审核无法保证任意设备上通过，也没有任何读数、模型生成或训练记录。

**练习（不附答案）：** ① 写出 L=5,W=3,n=2 的每轮 context 长度和返回 shape；② 解释为何 sample 的 key 必须随轮次更新；③ 说明 why context cropping 不能替代 KV cache。

本节材料初稿，实际修正 0/2；按用户新授权自动继续下一小节。


**工作副本的导入位置（Working-directory discovery）**：下列代码在仓库 root、`notebooks/` 或 `notebooks/work/` 启动 kernel 时，均从 `Path.cwd()` 向上查找同时含 `pyproject.toml` 与 `src/minigpt/__init__.py` 的目录，再将其 `src` 加入 `sys.path`。这不是依赖单一 cwd 的硬编码路径；如果 kernel 在仓库树外启动，将明确抛出 `FileNotFoundError`。这段逻辑只负责定位代码，**没有运行课程代码，也不是自动安装**。

### 23.4 精确的生成轴与两条选择策略
`_last_logits` 的源码先得到 `logits = model(context)`，shape **`(1,T,V)`**，然后用 **`logits[:, -1, :] → last (1,V)`** 取最后一个 query 位置的全部词表分数；`last` 已经是二维数组，不能再对它执行 `last[:, -1, :]`。

`history` 保存完整 prompt + 已生成 token；每轮只把 `context = history[:, -block_size:]` 交给模型，新的 `token (1,1)` 要沿时间轴追加到 **history**，不是覆盖为短 context。当前只实现 **greedy argmax** 和 **全词表（full-vocabulary）temperature categorical sampling**；**没有 top-k、top-p 或 EOS**。Step 21.4 的 top-k 教材函数没有被集成进 `generate_sample`。

抽样使用 `next_key, draw_key = mx.random.split(key)`：`draw_key` 只供本轮 `mx.random.categorical`，而 `key = next_key` 留给下一轮并随 history 返回，不复用本轮抽样 key。两条生成函数都在 `model.eval()` 后将循环包进 `try/finally`，退出或抛异常时 `model.train(was_training)` 恢复调用前的模式；这不表示已成功生成，也不是发生错误时回滚 history。

**2026-11-18 独立复核首次教学修订：1/2；未执行。**
